# CUDA SGM: GPU tests and benchmark

Run in Colab with a T4 (Runtime, then Change runtime type, then T4 GPU). This notebook:

1. builds the `stereo_depth_cuda` extension against Colab's PyTorch,
2. runs the standalone kernel self-test and `pytest -m gpu`,
3. compares the CUDA engine with the NumPy reference on Middlebury Cones and times both.

Set `REF` to the branch or tag you want to test.

In [ ]:
REF = "main"
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!rm -rf stereo-depth && git clone -q --depth 1 -b {REF} https://github.com/PhillysTheBest/stereo-depth
!pip install -q --no-build-isolation ./stereo-depth/cuda
!pip install -q ./stereo-depth/backend pytest pillow

## Kernel self-test (no PyTorch) and GPU tests

In [ ]:
!make -C stereo-depth/cuda test
!cd stereo-depth/backend && python -m pytest -m gpu -v

## Middlebury Cones (quarter size)

Done when CUDA is within 1% bad pixels of the CPU reference and faster.
Ground truth `disp2.png` stores disparity times 4, with 0 meaning unknown.

In [ ]:
import urllib.request

import numpy as np
from PIL import Image
from stereo_depth.engines import CudaSGMEngine, SGMCpuEngine
from stereo_depth.sgm import SgmParams

BASE = "https://vision.middlebury.edu/stereo/data/scenes2003/newdata/cones/"


def fetch(name: str) -> np.ndarray:
    urllib.request.urlretrieve(BASE + name, name)
    return np.asarray(Image.open(name).convert("RGB" if name.startswith("im") else "L"))


left, right, gt = fetch("im2.png"), fetch("im6.png"), fetch("disp2.png") / 4.0
params = SgmParams(max_disp=64)
cpu_engine, gpu_engine = SGMCpuEngine(params), CudaSGMEngine(params)
assert gpu_engine.is_available()

cpu = cpu_engine.predict(left, right)
gpu_engine.predict(left, right)  # warm-up
gpu_runs = [gpu_engine.predict(left, right) for _ in range(20)]
gpu = gpu_runs[-1]
gpu_ms = float(np.median([r.runtime_ms for r in gpu_runs]))

known = gt > 0
results = {
    "cpu_vs_cuda_bad1_pct": 100 * float((np.abs(cpu.values - gpu.values) > 1).mean()),
    "cpu_bad2_pct": 100 * float((np.abs(cpu.values - gt) > 2)[known].mean()),
    "cuda_bad2_pct": 100 * float((np.abs(gpu.values - gt) > 2)[known].mean()),
    "cpu_ms": cpu.runtime_ms,
    "cuda_ms": gpu_ms,
    "speedup": cpu.runtime_ms / gpu_ms,
}
for key, value in results.items():
    print(f"{key:>22}: {value:.2f}")
assert results["cpu_vs_cuda_bad1_pct"] < 1.0
assert results["cuda_ms"] < results["cpu_ms"]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (title, img) in zip(
    axes, [("left", left), ("ground truth", gt), ("CPU SGM", cpu.values), ("CUDA SGM", gpu.values)]
):
    ax.imshow(img, cmap=None if img.ndim == 3 else "magma")
    ax.set_title(title)
    ax.axis("off")
plt.show()

## Speed at 1280x720 with 128 disparities

In [ ]:
rng = np.random.default_rng(0)
big_left = rng.integers(0, 256, (720, 1280), dtype=np.uint8)
big_right = np.roll(big_left, -20, axis=1)
big = CudaSGMEngine(SgmParams(max_disp=128))
big.predict(big_left, big_right)
big_ms = float(np.median([big.predict(big_left, big_right).runtime_ms for _ in range(10)]))
results["cuda_720p_ms"] = big_ms
print(f"720p, 128 disparities: {big_ms:.1f} ms (includes host to device copies)")

Optional: log to Weights & Biases if `WANDB_API_KEY` is set in Colab secrets.

In [ ]:
import os

if os.environ.get("WANDB_API_KEY"):
    import wandb

    run = wandb.init(project="stereo-depth", job_type="cuda-sgm-bench", config={"ref": REF})
    run.log(results)
    run.finish()